#  Single-Qubit State Tomography using Maximum likelihood estimation (RρR algorithm)
*Reconstructing an unknown quantum state from measurements.*

This notebook demonstrates the complete process of single-qubit state tomography using RρR algorithm:


In [12]:
#imports 
from qutip import qeye
import numpy as np
import matplotlib.pyplot as plt
from qutip import about, basis, sigmax, sigmay, sigmaz, qeye, Qobj, fidelity, ket2dm
from qutip.measurement import measure
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display, Math, Latex, Markdown, HTML

# Set up for inline plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12


Define a function to check whether the reconstructed density matrix represents a physical state.

In [3]:
def check_physicality(rho, tol=1e-6):
    """
    Check if a density matrix is physical.
    
    Returns:
        dict: Results of various physicality checks
    """
    # Check 1: Hermiticity
    is_herm = rho.isherm
    
    # Check 2: Trace = 1
    trace = rho.tr()
    trace_ok = abs(trace - 1) < tol
    
    # Check 3: Positive semidefinite (eigenvalues ≥ 0)
    eigenvalues = rho.eigenenergies()
    pos_ok = np.all(eigenvalues >= -tol)
    
    # Check 4: Purity (should be ≤ 1)
    purity = (rho * rho).tr()
    purity_ok = purity <= 1 + tol
    
    # Check 5: Bloch vector length (should be ≤ 1)
    x = np.real((rho * sigmax()).tr())
    y = np.real((rho * sigmay()).tr())
    z = np.real((rho * sigmaz()).tr())
    bloch_length = np.sqrt(x**2 + y**2 + z**2)
    bloch_ok = bloch_length <= 1 + tol
    
    return {
        'is_physical': is_herm and trace_ok and pos_ok and purity_ok and bloch_ok,
        'is_hermitian': is_herm,
        'trace': trace,
        'trace_ok': trace_ok,
        'eigenvalues': eigenvalues,
        'positive_ok': pos_ok,
        'purity': purity,
        'purity_ok': purity_ok,
        'bloch_vector': (x, y, z),
        'bloch_length': bloch_length,
        'bloch_ok': bloch_ok
    }

Define a function to simulate measurements.

In [4]:
def simulate_measurements(target_state, n_shots=1000, seed=None):
    """
    Simulate measurements using Pauli matrices (CORRECT for measure()).
    
    Args:
        target_state: The quantum state to measure
        n_shots: Number of measurements per basis
        seed: Random seed
    
    Returns:
        plus_counts: Dictionary with counts of +1 outcomes
        measurement_results: Raw measurement results
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Use PAULI MATRICES for measurement (these have eigenvalues ±1)
    operators = {
        'X': sigmax(),
        'Y': sigmay(),
        'Z': sigmaz()
    }
    
    measurement_results = {'X': [], 'Y': [], 'Z': []}
    plus_counts = {}
    
    print(f"\nSimulating {n_shots} measurements in each basis...")
    
    for basis_name, operator in operators.items():
        print(f"\n{basis_name}-basis measurements:")
        
        # Perform measurements
        for shot in range(n_shots):
            fresh_state = target_state.copy()
            result, collapsed_state = measure(fresh_state, operator)
            measurement_results[basis_name].append(result)
        
        # Calculate statistics
        results_array = np.array(measurement_results[basis_name])
        count_plus = np.sum(results_array == 1)
        count_minus = n_shots - count_plus
        prob_plus = count_plus / n_shots
        
        plus_counts[basis_name] = count_plus
        
        print(f"  Counts: +1: {count_plus:5d}, -1: {count_minus:5d}")
        print(f"  P(+) = {prob_plus:.4f}")
    
    return plus_counts, measurement_results

Define the RρR algorithm:

In [5]:
def rpr_algorithm(measurement_operators, counts, dim=2, max_iter=1000, tol=1e-8, dilution=0.5):
    """
    RρR algorithm - works for BOTH pure and mixed states.
    
    The update is: ρ ← R ρ R
    
    Args:
        measurement_operators: List of POVM elements (Qobj)
        counts: List of observed counts
        dim: Hilbert space dimension
        max_iter: Maximum iterations
        tol: Convergence tolerance
        dilution: Step size (0 < dilution ≤ 1) - default 0.5 works well
    
    Returns:
        rho: Reconstructed density matrix (Qobj)
    """
    # Start from maximally mixed state
    rho = qeye(dim) / dim
    
    for iteration in range(max_iter):
        rho_old = rho.copy()
        
        # Build R matrix = Σ (n_k / Tr(ρ M_k)) M_k
        R = Qobj(np.zeros((dim, dim), dtype=complex), dims=rho.dims)
        
        for M_k, n_k in zip(measurement_operators, counts):
            prob = np.real((rho * M_k).tr())
            if prob > 1e-12:  # Avoid division by zero
                R += (n_k / prob) * M_k
        
        # RρR update: ρ_new = R ρ R
        rho_new = R * rho * R
        trace_new = rho_new.tr()
        
        if abs(trace_new) > 1e-12:
            rho_new = rho_new / trace_new
        else:
            # Fallback if something went wrong
            rho_new = qeye(dim) / dim
        
        # Diluted update for stability
        rho = (1 - dilution) * rho + dilution * rho_new
        rho = rho / rho.tr()  # Re-normalize after mixing
        
        # Check convergence
        diff = (rho - rho_old).norm()
        if diff < tol:
            break
    
    return rho


Single-qubit state tomography for pure state using EM algorithm: 

$$ \rho = |+i\rangle\langle +i| = \frac{1}{2} \begin{pmatrix} 1 & -i \\ i & 1 \end{pmatrix} $$

In [6]:
# Step 1: Create a target state
target_state = (basis(2, 0) + 1j * basis(2, 1)).unit()  # |+i⟩
print("Target state: |+i⟩ = (|0⟩ + i|1⟩)/√2")

# Step 2: Simulate measurements using PAULI MATRICES
n_shots = 1000
plus_counts, raw_results = simulate_measurements(target_state, n_shots=n_shots, seed=42)

# Step 3: Prepare data for EM algorithm using PROJECTORS
I = qeye(2)

# Create projectors (for EM algorithm)
M_X = (I + sigmax()) / 2   # |+⟩⟨+| projector
M_Y = (I + sigmay()) / 2   # |+i⟩⟨+i| projector
M_Z = (I + sigmaz()) / 2   # |0⟩⟨0| projector

# We need ALL outcomes for EM (both plus and minus)
measurement_ops = [
    M_X, I - M_X,  # X-basis: plus and minus
    M_Y, I - M_Y,  # Y-basis: plus and minus
    M_Z, I - M_Z   # Z-basis: plus and minus
]

# Create full counts array [X+, X-, Y+, Y-, Z+, Z-]
full_counts = [
    plus_counts['X'], n_shots - plus_counts['X'],
    plus_counts['Y'], n_shots - plus_counts['Y'],
    plus_counts['Z'], n_shots - plus_counts['Z']
]

print(f"\nFull counts for MLE: {full_counts}")




Target state: |+i⟩ = (|0⟩ + i|1⟩)/√2

Simulating 1000 measurements in each basis...

X-basis measurements:
  Counts: +1:   497, -1:   503
  P(+) = 0.4970

Y-basis measurements:
  Counts: +1:  1000, -1:     0
  P(+) = 1.0000

Z-basis measurements:
  Counts: +1:   502, -1:   498
  P(+) = 0.5020

Full counts for MLE: [np.int64(497), np.int64(503), np.int64(1000), np.int64(0), np.int64(502), np.int64(498)]


In [7]:
# Step 4: Run the algorithm
print("\nRunning EM algorithm...")
rho_em = rpr_algorithm(measurement_ops, full_counts)
print(rho_em)
#Check whether reconstructed density matrix is a physical state
check_physicality(rho_em)


Running EM algorithm...
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=True
Qobj data =
[[ 0.50133333+1.59661024e-20j -0.002     -4.99994211e-01j]
 [-0.002     +4.99994211e-01j  0.49866667+2.98169784e-21j]]


{'is_physical': np.True_,
 'is_hermitian': True,
 'trace': 1.0,
 'trace_ok': True,
 'eigenvalues': array([1.07334394e-08, 9.99999989e-01]),
 'positive_ok': np.True_,
 'purity': 0.9999999785331215,
 'purity_ok': True,
 'bloch_vector': (-0.003999998232925165,
  0.9999884229418318,
  0.002666657587245269),
 'bloch_length': np.float64(0.9999999785331212),
 'bloch_ok': np.True_}

In [8]:
target_dm = ket2dm(target_state)
fidelity_value = fidelity(target_dm, rho_em)

print(f"Fidelity between target and reconstructed state: {fidelity_value:.8f}")
print(f"Percentage overlap: {fidelity_value*100:.4f}%")

Fidelity between target and reconstructed state: 0.99999711
Percentage overlap: 99.9997%


Testing the algorithm for multiple states with simulated measurements.

In [25]:

def test_rpr_with_simulated_measurements():
    """Test RρR algorithm using simulated measurement data with statistical noise."""
    print("="*70)
    print("TESTING RρR ALGORITHM WITH SIMULATED MEASUREMENTS")
    print("="*70)
    
    # Setup measurement operators for RρR (POVM elements, not Paulis)
    I = qeye(2)
    M_X = (I + sigmax()) / 2
    M_Y = (I + sigmay()) / 2
    M_Z = (I + sigmaz()) / 2
    measurement_ops = [M_X, I-M_X, M_Y, I-M_Y, M_Z, I-M_Z]
    
    # Test states
    test_states = {
        "Pure |0⟩": ket2dm(basis(2,0)),
        "Pure |+⟩": ket2dm((basis(2,0)+basis(2,1)).unit()),
        "Pure |+i⟩": ket2dm((basis(2,0)+1j*basis(2,1)).unit()),
        "Mixed (70/30)": 0.7*ket2dm(basis(2,0)) + 0.3*ket2dm(basis(2,1)),
        "Mixed (90/10)": 0.9*ket2dm((basis(2,0)+basis(2,1)).unit()) + 0.1*ket2dm((basis(2,0)-basis(2,1)).unit()),
        "Maximally Mixed": I/2,
    }
    
    n_shots = 1000
    print(f"\n{'State':<20} {'Fidelity':<12} {'Physical':<9} {'Result':<6}")
    print("-"*50)
    
    results = {}
    for name, true_rho in test_states.items():
        # Simulate measurements with verbose=False to suppress output
        plus_counts, _ = simulate_measurements(
            true_rho, 
            n_shots=n_shots, 
            seed=42,
            verbose=False  # This suppresses the detailed output
        )
        
        # Convert Pauli measurement results to POVM counts for RρR
        povm_counts = []
        for basis_name in ['X', 'Y', 'Z']:
            count_plus = plus_counts[basis_name]
            count_minus = n_shots - count_plus
            povm_counts.extend([count_plus, count_minus])
        
        # Run RρR
        rho = rpr_algorithm(measurement_ops, povm_counts, max_iter=500, dilution=0.5)
        
        # Check physicality
        phys = check_physicality(rho)
        
        # Metrics
        fid = np.real(fidelity(true_rho, rho))
        success = fid > 0.99 and phys['is_physical']
        
        results[name] = {'fidelity': fid, 'physical': phys['is_physical']}
        
        phys_symbol = "✅" if phys['is_physical'] else "❌"
        success_symbol = "✅" if success else "❌"
        print(f"{name:<20} {fid:<12.8f} {phys_symbol:<9} {success_symbol:<6}")
    
    # Summary
    print(f"\n✅ Success rate: {sum(1 for r in results.values() if r['fidelity']>0.99 and r['physical'])}/{len(results)}")
    
    return results

if __name__ == "__main__":
    results = test_rpr_with_simulated_measurements()

TESTING RρR ALGORITHM WITH SIMULATED MEASUREMENTS

State                Fidelity     Physical  Result
--------------------------------------------------
Pure |0⟩             0.99994109   ✅         ✅     
Pure |+⟩             0.99994221   ✅         ✅     
Pure |+i⟩            0.99999711   ✅         ✅     
Mixed (70/30)        0.99986481   ✅         ✅     
Mixed (90/10)        0.99978836   ✅         ✅     
Maximally Mixed      0.99986545   ✅         ✅     

✅ Success rate: 6/6
